In [0]:
spark.catalog.setCurrentCatalog("futebol_analytics")

In [0]:
from pyspark.sql.functions import (
    avg,
    col,
    count,
    current_timestamp,
    dense_rank,
    monotonically_increasing_id,
    round,
    sum,
    when,
    year,
)
from pyspark.sql.window import Window

spark.catalog.setCurrentCatalog("futebol_analytics")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# ==============================================================================
# 1. DIMENSÃO JOGADORES (dim_jogadores)
# ==============================================================================
print("Construindo dim_jogadores...")

df_dim_jogadores = (
    spark.table("silver.players")
    .select(
        col("player_id"),
        col("player_name"),
        col("player_name_clean"),
        col("date_of_birth"),
        col("country_of_citizenship"),
        col("position"),
        col("sub_position"),
        col("foot"),
        col("height_in_cm"),
        col("current_club_id"),
        col("market_value_in_eur"),
        col("highest_market_value_in_eur"),
    )
    .withColumn("_created_at", current_timestamp())
)

df_dim_jogadores.write.format("delta").mode("overwrite").saveAsTable(
    "gold.dim_jogadores"
)
print(
    f"[OK] gold.dim_jogadores criada com"
    f" {spark.table('gold.dim_jogadores').count():,} atletas."
)

# ==============================================================================
# 2. DIMENSÃO CLUBES (dim_clubes)
# ==============================================================================
print("\nConstruindo dim_clubes...")

df_dim_clubes = (
    spark.table("silver.clubs")
    .select(
        col("club_id"),
        col("club_code"),
        col("club_name"),
        col("domestic_competition_id"),
        col("total_market_value"),
        col("stadium_name"),
    )
    .withColumn("_created_at", current_timestamp())
)

df_dim_clubes.write.format("delta").mode("overwrite").saveAsTable(
    "gold.dim_clubes"
)
print(
    f"[OK] gold.dim_clubes criada com"
    f" {spark.table('gold.dim_clubes').count():,} clubes."
)

# ==============================================================================
# 3. DIMENSÃO TEMPO (dim_tempo)
# ==============================================================================
print("\nConstruindo dim_tempo...")

anos_temporadas = [(2022, "2021/2022"), (2023, "2022/2023"), (2024, "2023/2024"), (2025, "2024/2025")]

df_dim_tempo = (
    spark.createDataFrame(anos_temporadas, ["ano", "temporada"])
    .withColumn("sk_tempo", monotonically_increasing_id())
    .withColumn("_created_at", current_timestamp())
)

df_dim_tempo.write.format("delta").mode("overwrite").saveAsTable(
    "gold.dim_tempo"
)
print(
    f"[OK] gold.dim_tempo criada com"
    f" {spark.table('gold.dim_tempo').count()} safras temporais."
)

In [0]:
from pyspark.sql.functions import (
    avg,
    col,
    count,
    current_timestamp,
    lower,
    monotonically_increasing_id,
    round,
    sum,
    trim,
    when,
    year,
)

spark.catalog.setCurrentCatalog("futebol_analytics")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

print("Gerando tabela fato de transferências com histórico temporal (t e t-1)...")

# 1. Base consolidada de partidas por atleta e ano (Appearances)
df_app_anual = (
    spark.table("futebol_analytics.silver.appearances")
    .withColumn("ano_partida", year("date"))
    .groupBy("player_id", "ano_partida")
    .agg(
        sum("minutes_played").alias("minutos_jogados"),
        count("appearance_id").alias("total_jogos"),
        (sum("goals") + sum("assists")).alias("gols_assistencias"),
    )
)

# Separação explícita de nomes para evitar referências ambíguas nos joins
df_app_t = df_app_anual.select(
    col("player_id").alias("player_id_t"),
    col("ano_partida").alias("ano_partida_t"),
    col("minutos_jogados").alias("minutos_jogados_ano_t"),
    col("total_jogos").alias("total_jogos_ano_t"),
    col("gols_assistencias").alias("gols_assistencias_ano_t"),
)

df_app_prev = df_app_anual.select(
    col("player_id").alias("player_id_prev"),
    col("ano_partida").alias("ano_partida_prev"),
    col("minutos_jogados").alias("minutos_jogados_t_minus_1"),
    col("total_jogos").alias("total_jogos_t_minus_1"),
    col("gols_assistencias").alias("gols_assistencias_t_minus_1"),
)

# 2. Base de notas da FotMob agregadas por atleta e ano
df_fot_anual = (
    spark.table("futebol_analytics.silver.fotmob_ratings")
    .groupBy(
        lower(trim(col("player_name_clean"))).alias("fot_name_clean"),
        col("match_year").alias("fot_match_year"),
    )
    .agg(
        round(avg("rating"), 2).alias("nota_media_ano_t"),
        count("rating").alias("partidas_avaliadas_ano_t"),
    )
)

# 3. Base de transferências na janela analítica (2022 a 2025)
df_transfers = (
    spark.table("futebol_analytics.silver.transfers")
    .filter(col("transfer_year").between(2022, 2025))
    .withColumn("tr_name_clean", lower(trim(col("player_name_clean"))))
)

# 4. Joins dimensionais e métricas de sucesso
df_fato_completa = (
    df_transfers
    # Join temporada t (ano da contratação)
    .join(
        df_app_t,
        (col("player_id") == col("player_id_t"))
        & (col("transfer_year") == col("ano_partida_t")),
        how="left",
    )
    # Join temporada t-1 (ano imediatamente anterior)
    .join(
        df_app_prev,
        (col("player_id") == col("player_id_prev"))
        & ((col("transfer_year") - 1) == col("ano_partida_prev")),
        how="left",
    )
    # Join avaliações da FotMob no ano t
    .join(
        df_fot_anual,
        (col("tr_name_clean") == col("fot_name_clean"))
        & (col("transfer_year") == col("fot_match_year")),
        how="left",
    )
    .withColumn("sk_fato_transferencia", monotonically_increasing_id())
    .withColumn(
        "diferenca_agio_desagio",
        round(col("transfer_fee") - col("market_value_in_eur"), 2),
    )
    .fillna(
        {
            "minutos_jogados_ano_t": 0,
            "total_jogos_ano_t": 0,
            "gols_assistencias_ano_t": 0,
            "minutos_jogados_t_minus_1": 0,
            "total_jogos_t_minus_1": 0,
            "gols_assistencias_t_minus_1": 0,
        }
    )
    # Critérios de Sucesso
    .withColumn(
        "flag_sucesso_fisico",
        when(col("minutos_jogados_ano_t") >= 1200, 1).otherwise(0),
    )
    .withColumn(
        "flag_sucesso_tecnico",
        when(col("nota_media_ano_t") >= 6.8, 1).otherwise(0),
    )
    .withColumn(
        "flag_sucesso_consolidado",
        when(
            (col("minutos_jogados_ano_t") >= 1200)
            & (col("nota_media_ano_t") >= 6.8),
            1,
        ).otherwise(0),
    )
    .withColumn("_created_at", current_timestamp())
    .select(
        col("sk_fato_transferencia"),
        col("player_id"),
        col("from_club_id"),
        col("to_club_id"),
        col("transfer_date"),
        col("transfer_year").alias("ano_transferencia"),
        col("transfer_fee"),
        col("market_value_in_eur"),
        col("diferenca_agio_desagio"),
        col("minutos_jogados_t_minus_1"),
        col("total_jogos_t_minus_1"),
        col("gols_assistencias_t_minus_1"),
        col("minutos_jogados_ano_t"),
        col("total_jogos_ano_t"),
        col("gols_assistencias_ano_t"),
        col("nota_media_ano_t"),
        col("partidas_avaliadas_ano_t"),
        col("flag_sucesso_fisico"),
        col("flag_sucesso_tecnico"),
        col("flag_sucesso_consolidado"),
        col("_created_at"),
    )
)

# 5. Sobrescrita limpa da tabela fato
spark.sql("DROP TABLE IF EXISTS futebol_analytics.gold.fato_transferencias_desempenho")

df_fato_completa.write.format("delta").mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("futebol_analytics.gold.fato_transferencias_desempenho")

total_registros = spark.table("futebol_analytics.gold.fato_transferencias_desempenho").count()
print(f"[SUCESSO] Tabela fato gerada com {total_registros:,} contratações.")

In [0]:
total_silver = (
    spark.table("futebol_analytics.silver.transfers")
    .filter("transfer_year BETWEEN 2022 AND 2025")
    .count()
)
total_gold = spark.table(
    "futebol_analytics.gold.fato_transferencias_desempenho"
).count()

print(f"Total de transferências na Silver (2022-2025): {total_silver:,}")
print(f"Total de registros na Gold (Tabela Fato):       {total_gold:,}")

assert total_silver == total_gold, "Houve multiplicação indevida de linhas!"
print(
    "[OK] Integridade validada: nenhuma linha duplicada no cruzamento"
    " dimensional."
)